# Exp 24: Head erase on non-VIP cats

Card: [exp24_head_erase.md](../../experiments/exp24_head_erase.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, MLFLOW_URI, ROOT, TRAIN_DIR
from core.cv import VIP, Config, CrossValidator, CVResult
from core.data import FACE_BOXES, HEAD_BOXES, read_boxes, read_train
from core.gradcam import GradCamFigures, compare, random_box

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-head-erase-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_BOXES = 2  # images in box figure
    N_ROWS = 4  # positives in recall test
    N_SAMPLE = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_BOXES = 16
    N_ROWS = None  # all generic positives with head box
    N_SAMPLE = 16  # exp19 sample
SEED = 0  # exp19 sample, random boxes
CLS = "Cat"
FOLD = 0
MAX_AREA = 0.3  # `EraseBoxes` skips larger boxes
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp24_head_erase" / "resnet18.toml", **OVERRIDES
)
OLD = {
    "exp16": Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml"),
    "exp20": Config.from_toml(
        CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml"
    ),
    "exp21": Config.from_toml(CONFIG_DIR / "exp21_face_erase" / "resnet18.toml"),
}
CONDITIONS = ["clean", "random box", "head erased"]
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp24_head_erase"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {
    "exp16": "#eb6834",
    "exp20": "#2a78d6",
    "exp21": "#1baf7a",
    "exp24": "#7c5cd6",
}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## Head boxes


In [ ]:
faces, heads = read_boxes(FACE_BOXES), read_boxes(
    HEAD_BOXES
)  # heads: `python -m core.boxes --head`
train = read_train()
face = {
    i: dict(faces.get(image_id, [])).get(CLS)
    for i, image_id in train["image_id"].items()
}
head = {
    i: dict(heads.get(image_id, [])).get(CLS)
    for i, image_id in train["image_id"].items()
}
generic = [
    i
    for i in train.index[(train[CLS] == 1) & ~train[VIP].any(axis=1)]
    if head[i] is not None
]
skipped = np.mean(
    [(x1 - x0) * (y1 - y0) > MAX_AREA for x0, y0, x1, y1 in (head[i] for i in generic)]
)
figures = {}
sample = (
    train.loc[generic].query("fold != @FOLD").sample(N_BOXES, random_state=SEED).index
)
fig, axes = plt.subplots(2, N_BOXES // 2, figsize=(1.1 * N_BOXES, 4.8), squeeze=False)
for ax, i in zip(axes.flat, sample):
    ax.imshow(
        Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg")
        .convert("RGB")
        .resize((224, 224))
    )
    for (x0, y0, x1, y1), color in [(face[i], "#eb6834"), (head[i], "#2a78d6")]:
        ax.add_patch(
            Rectangle(
                (x0 * 224, y0 * 224),
                (x1 - x0) * 224,
                (y1 - y0) * 224,
                fill=False,
                color=color,
                lw=2,
            )
        )
for ax in axes.flat:
    ax.set_axis_off()
fig.suptitle(
    f"{CLS} · face box (orange), head box (blue) · fold {FOLD} train part, generic · "
    f"{skipped:.1%} of head boxes above {MAX_AREA:.0%} of image, not erased",
    color="#52514e",
)
fig.tight_layout()
figures[f"{PREFIX}_head_boxes.png"] = fig
plt.show()

## Training


In [ ]:
results = {name: CVResult.load(config) for name, config in OLD.items()}
results["exp24"] = CrossValidator(CONFIG).run()

## Exam: recall without head


In [ ]:
cams = {
    name: GradCamFigures(config) for name, config in {**OLD, "exp24": CONFIG}.items()
}
rng = np.random.default_rng(SEED)
in_oof = set(cams["exp24"].oof.index)  # smoke OOF holds few rows
rows = [i for i in generic if train.loc[i, "fold"] in (0, 1) and i in in_oof][:N_ROWS]
boxes = {
    "clean": {},
    "random box": {i: random_box(head[i], rng) for i in rows},
    "head erased": {i: head[i] for i in rows},
}
recall = {
    (name, cond): (cam.probs(rows, CLS, boxes[cond]) >= cam.thresholds[CLS]).mean()
    for name, cam in cams.items()
    for cond in CONDITIONS
}
fig, ax = plt.subplots(figsize=(5, 3.8))
for name in cams:
    y = [recall[name, cond] for cond in CONDITIONS]
    ax.plot(CONDITIONS, y, marker="o", lw=2, color=SERIES[name], label=name)
    ax.annotate(
        f"{y[-1]:.3f}",
        (2, y[-1]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        fontsize=8,
        color=SERIES[name],
    )
ax.set(
    title=f"{CLS} ({len(rows)} generic positives)",
    ylabel="recall at own best-F1 threshold",
    xlim=(-0.2, 2.5),
)
ax.legend(loc="lower left", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_recall.png"] = fig
for key, value in recall.items():
    print(*key, f"{value:.3f}")
plt.show()

## Grad-CAM with head erased


In [ ]:
source = "exp24" if SMOKE else "exp16"  # exp19 rows drawn from exp16 OOF
sample = cams[source].positive_rows(CLS, N_SAMPLE, SEED)
erased = {i: head[i] for i in sample if head[i] is not None}
fig = compare(cams, CLS, sample, boxes=erased
fig.suptitle(f"{CLS} · head erased · validation (model held image out) · {len(sample)} labeled positives", color="#52514e")
figures[f"{PREFIX}_grad_cam_cat.png"] = fig
plt.show()

## Precision-recall curves


In [ ]:
results.pop("exp16")  # guard against exp20, exp21
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with mlflow.start_run(
    experiment_id=client.get_experiment_by_name(CONFIG.experiment).experiment_id,
    run_name="summary",
):
    mlflow.log_param("val_mode", "fast")
    if SAVE_FIGURES:
        for fname, fig in figures.items():
            fig.savefig(FIGURES / fname, dpi=120)
            mlflow.log_figure(fig, fname)